# 00 · Inicio en Google Colab — Proyecto Estadísticas de YouTube

Este notebook prepara el entorno y ejecuta la etapa de datos de principio a fin. La explicación detallada de cada paso está en la guía del taller (`docs/guia_paso_a_paso.md` del repositorio).

**Antes de ejecutar nada:** *Archivo → Guardar una copia en Drive*. El notebook abierto desde el enlace del taller es de solo lectura; la copia queda en Google Drive y es la que el equipo edita y entrega.

Qué hace este notebook:

1. Descarga el código del taller (solo lectura) e instala lo que Colab no trae.
2. Monta Google Drive, donde se guardan los datos, los reportes y la configuración del equipo.
3. Carga las credenciales desde los **Secretos** de Colab (nunca escritas en el notebook).
4. Ejecuta el pipeline con la muestra, con Kaggle y con la YouTube Data API.
5. Muestra los reportes.

**Secretos necesarios** (panel izquierdo de Colab → *Secretos*; activar *Acceso del notebook* en cada uno):

| Secreto | Para qué | Dónde se obtiene |
| --- | --- | --- |
| `KAGGLE_API_TOKEN` (o `KAGGLE_USERNAME` y `KAGGLE_KEY`) | Descargar el dataset de Kaggle | kaggle.com/settings/api → *Generate New Token* (o *Create Legacy API Key*) |
| `YOUTUBE_API_KEY` | Extraer datos con la YouTube Data API v3 | Google Cloud Console → APIs y servicios → Credenciales |

No se necesita GPU para esta etapa: *Entorno de ejecución → Cambiar tipo de entorno de ejecución* → CPU.

In [ ]:
# @title 1 · Descargar el código del taller e instalar dependencias
# Repositorio público del taller (lo define el docente; no se modifica).
REPO_URL = "https://github.com/USUARIO/REPOSITORIO.git"
BRANCH = "main"

import os, subprocess, sys

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git " + args[0] + " falló:\n" + r.stderr[-500:])

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if "USUARIO/REPOSITORIO" in REPO_URL:
        raise ValueError("El repositorio del taller no está configurado; avisar al docente.")
    REPO_DIR = "/content/youtube-stats-nlp"
    if os.path.exists(REPO_DIR):  # actualiza a la última versión publicada por el docente
        _git("-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", BRANCH)
        _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
    else:
        _git("clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_DIR}/requirements-colab.txt"], check=True)
    os.chdir(REPO_DIR)
else:  # computador local: el notebook se abre desde la carpeta notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO_DIR)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Código del taller listo en", REPO_DIR)

In [ ]:
# @title 2 · Montar Drive y cargar secretos
CARPETA_DRIVE = "MyDrive/ytnlp-proyecto"  # @param {type:"string"}

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from ytnlp.colab import setup
info = setup(drive_folder=CARPETA_DRIVE)

## 3 · Verificar que todo funciona

Corre las pruebas automáticas y el pipeline completo sobre una muestra **sintética** (no requiere credenciales; sus resultados no sirven para conclusiones).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], check=False)
subprocess.run([sys.executable, "scripts/make_sample_data.py"], check=True)

from ytnlp.pipeline import run
run("sample")

## 4 · Configuración del equipo

Las decisiones del equipo se guardan en Drive (`config_equipo.yaml`) y se aplican en todas las sesiones y notebooks. El código del taller no se modifica.

- `TARGET`: variable objetivo (ver la Parte 6 de la guía).
- `COMENTARIOS_POR_VIDEO`: máximo de comentarios por video al extraer con la API.
- `RESTABLECER`: vuelve a la configuración base del taller.

In [ ]:
# @title Guardar configuración del equipo
TARGET = "engagement"  # @param ["engagement", "log_views", "log_likes", "perf_class"]
COMENTARIOS_POR_VIDEO = 100  # @param {type:"integer"}
RESTABLECER = False  # @param {type:"boolean"}

from ytnlp import config
if RESTABLECER:
    config.reset_overrides()
    print("Configuración base del taller restablecida.")
else:
    destino = config.set_overrides(target=TARGET, api__max_comments_per_video=COMENTARIOS_POR_VIDEO)
    print("Configuración guardada en", destino)
cfg = config.load_config()
print("target =", cfg["target"], "| comentarios por video =", cfg["api"]["max_comments_per_video"])

## 5 · Datos reales de Kaggle

Requiere el secreto `KAGGLE_API_TOKEN` (o el par heredado `KAGGLE_USERNAME` y `KAGGLE_KEY`). El dataset se descarga una sola vez a Drive.

In [ ]:
from ytnlp.data.kaggle_source import has_credentials
from ytnlp.pipeline import run
if not has_credentials():
    print("Faltan los secretos de Kaggle; ver la tabla al inicio del notebook.")
else:
    print(run("kaggle"))

## 6 · Ver los reportes

Los archivos quedan en la carpeta de Drive configurada (`reports/`), así que también se pueden abrir desde Google Drive.

In [ ]:
# @title Mostrar reportes
REPORTE = "quality_report.md"  # @param ["quality_report.md", "stats_report.md", "baseline_report.md"]
from IPython.display import Image, Markdown, display
from ytnlp.config import path

rep = path("reports")
texto = (rep / REPORTE).read_text(encoding="utf-8")
display(Markdown(texto.split("## Figuras")[0]))
if REPORTE == "stats_report.md":
    for img in sorted((rep / "figures").glob("*.png")):
        display(Image(filename=str(img), width=520))

## 7 · Ampliar los datos con la YouTube Data API

Toma los Video IDs del dataset de Kaggle y descarga, por video, estadísticas actualizadas, duración, suscriptores del canal y comentarios (hasta el máximo configurado en la sección 4). Cada llamada cuesta 1 unidad de cuota de las 10.000 diarias, así que conviene empezar con un límite pequeño. La extracción queda en Drive en `data/raw/api/<fecha>/`.

Ejecutar esta sección en días distintos genera una serie de tiempo y alimenta el historial de monitoreo (`reports/monitoring_history.csv`).

In [ ]:
# @title Extraer con la API
LIMITE_VIDEOS = 50  # @param {type:"integer"}
import os
if not os.getenv("YOUTUBE_API_KEY"):
    print("Falta el secreto YOUTUBE_API_KEY.")
else:
    from ytnlp.data.kaggle_source import load
    from ytnlp.data.youtube_api import extract
    ids = load()[0]["video_id"].dropna().unique().tolist()[:LIMITE_VIDEOS]
    carpeta = extract(ids)
    print("Extracción guardada en", carpeta)
    from ytnlp.pipeline import run
    run("api")

## Notas sobre Colab

- **La sesión es temporal.** Se desconecta tras un tiempo de inactividad o al llegar al máximo de horas, y el disco `/content` se borra. El código se vuelve a descargar en cada sesión; los datos, los reportes y la configuración siguen en Drive.
- **Al volver**, se abre la copia del notebook guardada en Drive y se ejecutan las celdas 1 y 2.
- **Trabajo en equipo:** la carpeta de Drive se comparte con el equipo; quien la recibe la agrega como *acceso directo en Mi unidad* y pone esa ruta en `CARPETA_DRIVE`. Cada integrante usa sus propios secretos.
- **Entrega:** se comparte con el docente la copia del notebook en Drive y la carpeta del proyecto.